# Document Embedding Visualization with ChromaDB — Student Exercise

Follow the demo and complete the gaps in this notebook. Use **guide.md** for hints and answers.

**Tasks:**
1. Load a PDF document and extract text
2. Chunk the text for embedding
3. Store embeddings in ChromaDB using sentence-transformers
4. Visualize the embedding space in 3D with Plotly

---

## 1. Prerequisites

If using Docker: `docker run -p 8888:8888 <image-name>` then open the URL with the token.

If using local setup: Run `./setup.sh` and select the **chroma_demo** kernel.

## 2. Import Libraries

**Exercise 1:** Add the missing import. We need to import `PCA` from scikit-learn for dimensionality reduction.

In [ ]:
print("Loading libraries...")

import chromadb
from chromadb.utils import embedding_functions
import PyPDF2
import plotly.express as px
# **FILL THE LINE**
import numpy as np
import pandas as pd

print("✅ All libraries imported successfully!")

## 3. PDF Loading Function

**Exercise 2:** Complete the PDF loading function. Fill in the missing class name for reading PDFs.

In [ ]:
def load_pdf_text(pdf_path: str) -> str:
    """Extract all text from a PDF file."""
    text_parts = []
    with open(pdf_path, 'rb') as f:
        # **FILL THE WORD**
        reader = PyPDF2.XXX(f)
        for page in reader.pages:
            text_parts.append(page.extract_text() or "")
    return "\n".join(text_parts)

print("✅ PDF loading function defined.")

## 4. Text Chunking Function

**Exercise 3:** Add the missing line that extracts and strips the chunk from the text.

In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list:
    """Split text into overlapping chunks."""
    if not text or not text.strip():
        return []
    chunks = []
    start = 0
    step = chunk_size - overlap
    while start < len(text):
        end = start + chunk_size
        # **FILL THE LINE**
        if chunk:
            chunks.append(chunk)
        start += step
    return chunks

print("✅ Chunking function defined.")

## 5. ChromaDB Setup

**Exercise 4:** Fill in the embedding model name. We use a popular lightweight model: `all-MiniLM-L6-v2`.

In [ ]:
client = chromadb.PersistentClient(path="./chroma_data")

print("Loading embedding model...")
# **FILL THE WORD**
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="FILL_MODEL_NAME")

collection_name = "doc_embeddings"
try:
    collection = client.get_collection(name=collection_name, embedding_function=ef)
    print(f"📂 Using existing collection: {collection_name}")
except Exception:
    collection = client.create_collection(name=collection_name, embedding_function=ef)
    print(f"📂 Created new collection: {collection_name}")

print("✅ ChromaDB setup complete.")

## 6. Main Processing: Load PDF, Chunk, and Add to ChromaDB

**Exercise 5:** Set the PDF path. Use `use_2025_budget.pdf` (included in Docker) or your own PDF.

In [ ]:
# **FILL THE WORD**
PDF_PATH = "FILL_PDF_PATH"

full_text = load_pdf_text(PDF_PATH)
print(f"📄 Loaded {len(full_text):,} characters from PDF.")

chunks = chunk_text(full_text, chunk_size=500, overlap=50)
print(f"📦 Created {len(chunks)} chunks.")

try:
    client.delete_collection(collection_name)
    collection = client.create_collection(name=collection_name, embedding_function=ef)
except Exception:
    pass

ids = [f"chunk_{i}" for i in range(len(chunks))]
metadatas = [{"chunk_index": i, "preview": chunk[:80] + "..." if len(chunk) > 80 else chunk} for i, chunk in enumerate(chunks)]

collection.add(
    documents=chunks,
    ids=ids,
    metadatas=metadatas
)

print(f"✅ Added {len(chunks)} chunks to ChromaDB collection '{collection_name}'.")

## 7. Plotly 3D Visualization

**Exercise 6:** Complete the visualization. Fill in the missing parameter for `collection.get()` — we need to include `"embeddings"`, `"documents"`, and `"metadatas"`.

In [ ]:
# **FILL THE WORD**
results = collection.get(include=[])
embeddings = np.array(results["embeddings"])
documents = results["documents"]
metadatas = results["metadatas"] or [{}] * len(documents)

pca = PCA(n_components=3)
reduced = pca.fit_transform(embeddings)

df = pd.DataFrame({
    "x": reduced[:, 0],
    "y": reduced[:, 1],
    "z": reduced[:, 2],
    "chunk_index": [m.get("chunk_index", i) for i, m in enumerate(metadatas)],
    "preview": [doc[:100] + "..." if len(doc) > 100 else doc for doc in documents],
})

fig = px.scatter_3d(
    df, x="x", y="y", z="z",
    color="chunk_index",
    color_continuous_scale="Turbo",
    hover_data=["chunk_index", "preview"],
    title="Document Chunk Embeddings (PCA 3D)",
)
fig.show()

print("✅ Plotly 3D visualization complete!")